<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/Module1_Labs(v2)/Lab4_TwoQubit_Gates_RZZ.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# QOS Lab 4 — Two-Qubit Gates & the R_ZZ Operator
### Quantum Optimization and Simulation | Cleveland State University
**Instructor:** Prof. Chansu Yu | Washkewicz College of Engineering

---
## Learning Objectives
1. Understand two-qubit states via the **tensor product** ($\otimes$)
2. Build and analyze the **CNOT** gate (entanglement)
3. Understand the **ZZ operator**: correlated phase flips based on qubit parity
4. Build the **$R_{ZZ}(\gamma)$** operator using CNOT–$R_Z$–CNOT decomposition
5. See how $R_{ZZ}$ encodes the **cut value of a single graph edge** into phase
6. *(Optional)* Run a $R_{ZZ}$ circuit on **real IBM quantum hardware**

---
### 📖 Background: Two-Qubit Systems

A two-qubit system has **4 basis states**: $|00\rangle, |01\rangle, |10\rangle, |11\rangle$.

The combined state of qubit A and qubit B:
$$|\psi_{AB}\rangle = |\psi_A\rangle \otimes |\psi_B\rangle$$

The **ZZ operator** acts on both qubits simultaneously:
$$ZZ|x_0x_1\rangle = (-1)^{x_0 \oplus x_1}|x_0x_1\rangle$$

- Same group ($x_0 = x_1$): eigenvalue $+1$ (no phase flip)
- Different group ($x_0 \neq x_1$): eigenvalue $-1$ (phase flip)

The **$R_{ZZ}(\gamma)$ rotation** extends this to a continuous phase:
$$R_{ZZ}(\gamma)|x_0x_1\rangle = e^{-i\frac{\gamma}{2}(-1)^{x_0\oplus x_1}}|x_0x_1\rangle$$

**This is exactly what we need for Max-Cut!** Edge $(i,j)$ is cut when $x_i \neq x_j$.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, Operator

simulator = AerSimulator()

# Single-qubit gates
I1 = np.eye(2, dtype=complex)
X  = np.array([[0,1],[1,0]], dtype=complex)
Z  = np.array([[1,0],[0,-1]], dtype=complex)
H  = (1/np.sqrt(2)) * np.array([[1,1],[1,-1]], dtype=complex)

# Basis states
ket_0 = np.array([1,0], dtype=complex)
ket_1 = np.array([0,1], dtype=complex)
ket_00 = np.kron(ket_0, ket_0)
ket_01 = np.kron(ket_0, ket_1)
ket_10 = np.kron(ket_1, ket_0)
ket_11 = np.kron(ket_1, ket_1)

print("Setup complete.")
print(f"|00⟩ = {ket_00}")
print(f"|01⟩ = {ket_01}")
print(f"|10⟩ = {ket_10}")
print(f"|11⟩ = {ket_11}")

---
## Part 1: Two-Qubit States and the Tensor Product

In [ ]:
# ── 1.1  Apply H⊗H to |00⟩: uniform 2-qubit superposition ───────────────────
HH = np.kron(H, H)
state_after_HH = HH @ ket_00

print("H⊗H applied to |00⟩:")
print(np.round(state_after_HH, 4))
print("\nExpected: [0.5, 0.5, 0.5, 0.5] — equal superposition of |00⟩,|01⟩,|10⟩,|11⟩")
print("\nProbabilities:")
labels = ['|00⟩','|01⟩','|10⟩','|11⟩']
for i, (lbl, amp) in enumerate(zip(labels, state_after_HH)):
    print(f"  P({lbl}) = |{amp:.4f}|² = {abs(amp)**2:.4f}")

In [ ]:
# ── 1.2  The ZZ operator ─────────────────────────────────────────────────────
ZZ = np.kron(Z, Z)
print("ZZ matrix:")
print(ZZ)
print("\nDiagonal: [+1, -1, -1, +1]")
print("  |00⟩ → +1 (same group, NOT cut)")
print("  |01⟩ → -1 (different groups, CUT!)")
print("  |10⟩ → -1 (different groups, CUT!)")
print("  |11⟩ → +1 (same group, NOT cut)")

# Verify
print("\nVerification:")
for lbl, ket in zip(['|00⟩','|01⟩','|10⟩','|11⟩'],
                     [ket_00, ket_01, ket_10, ket_11]):
    result = ZZ @ ket
    eigenvalue = np.dot(ket, result)  # ⟨ket|ZZ|ket⟩
    print(f"  ZZ{lbl} = {result}   eigenvalue = {eigenvalue:+.0f}")

---
## Part 2: The CNOT Gate and Entanglement

In [ ]:
# ── 2.1  CNOT gate ───────────────────────────────────────────────────────────
# CNOT flips the target qubit when the control qubit is |1⟩
# |ab⟩ → |a, a⊕b⟩
CNOT = np.array([[1,0,0,0],
                 [0,1,0,0],
                 [0,0,0,1],
                 [0,0,1,0]], dtype=complex)

print("CNOT matrix:")
print(CNOT.real.astype(int))

print("\nCNOT truth table:")
for lbl, ket in zip(['|00⟩','|01⟩','|10⟩','|11⟩'],
                     [ket_00, ket_01, ket_10, ket_11]):
    result = CNOT @ ket
    out_idx = np.argmax(abs(result))
    out_lbl = ['|00⟩','|01⟩','|10⟩','|11⟩'][out_idx]
    print(f"  CNOT {lbl} → {out_lbl}")

In [ ]:
# ── 2.2  CNOT in Qiskit: Bell state ──────────────────────────────────────────
qc_bell = QuantumCircuit(2, 2)
qc_bell.h(0)        # superposition on qubit 0
qc_bell.cx(0, 1)    # CNOT: control=0, target=1
qc_bell.measure([0,1],[0,1])

print("Bell State Circuit:")
print(qc_bell.draw('text'))

result = simulator.run(transpile(qc_bell, simulator), shots=1000).result()
counts = result.get_counts()
print(f"\nCounts: {counts}")
print("Only |00⟩ and |11⟩: the qubits are entangled!")

---
## Part 3: The R_ZZ Operator — QAOA's Edge Gate

In [ ]:
# ── 3.1  R_ZZ(γ) matrix ──────────────────────────────────────────────────────
def Rzz_matrix(gamma):
    """
    R_ZZ(γ) = e^{-iγ/2 ZZ}
    Diagonal: [e^{-iγ/2}, e^{+iγ/2}, e^{+iγ/2}, e^{-iγ/2}]
    """
    a = np.exp(-1j*gamma/2)   # same group  (|00⟩,|11⟩) → NOT cut
    b = np.exp(+1j*gamma/2)   # diff group  (|01⟩,|10⟩) → CUT
    return np.diag([a, b, b, a])

gamma = np.pi/3
Rzz = Rzz_matrix(gamma)

print(f"R_ZZ(γ=π/3) diagonal:")
for i, (lbl, val) in enumerate(zip(['|00⟩','|01⟩','|10⟩','|11⟩'], np.diag(Rzz))):
    phase_deg = np.degrees(np.angle(val))
    print(f"  {lbl}: e^(i·{phase_deg:+.1f}°) = {val:.4f}")

print("\nInterpretation:")
print("  |00⟩ and |11⟩ (same group, NOT cut): rotate by -γ/2")
print("  |01⟩ and |10⟩ (diff group, IS  cut): rotate by +γ/2")
print("  Relative phase between cut/not-cut = γ")

In [ ]:
# ── 3.2  R_ZZ decomposition: CNOT–R_Z–CNOT ───────────────────────────────────
# The standard quantum circuit for R_ZZ(γ) is:
#   CNOT → R_Z(γ) on target → CNOT

def rzz_circuit(gamma, q0=0, q1=1):
    """Build R_ZZ(γ) circuit using CNOT-Rz-CNOT decomposition."""
    qc = QuantumCircuit(2)
    qc.cx(q0, q1)          # CNOT
    qc.rz(gamma, q1)       # R_Z(γ) on target
    qc.cx(q0, q1)          # CNOT
    return qc

gamma = np.pi/3
qc_rzz = rzz_circuit(gamma)
print("R_ZZ(γ) circuit:")
print(qc_rzz.draw('text'))

# Verify the circuit matches the R_ZZ matrix
rzz_matrix_from_circuit = Operator(qc_rzz).data
rzz_exact = Rzz_matrix(gamma)

print(f"\nCircuit matrix (diagonal):")
for i, val in enumerate(np.diag(rzz_matrix_from_circuit)):
    print(f"  [{i}] {val:.4f}")

print(f"\nMatrices match: {np.allclose(rzz_matrix_from_circuit, rzz_exact)}")

In [ ]:
# ── 3.3  Apply R_ZZ to uniform superposition ──────────────────────────────────
# Start with 2-qubit uniform superposition: (|00⟩+|01⟩+|10⟩+|11⟩)/2
# Apply R_ZZ(γ) — should tag |01⟩,|10⟩ (the "cut" states) with extra phase

gamma = np.pi/3

qc_full = QuantumCircuit(2)
qc_full.h(0); qc_full.h(1)    # uniform superposition
qc_full.cx(0, 1)               # R_ZZ decomposition
qc_full.rz(gamma, 1)
qc_full.cx(0, 1)

sv = Statevector(qc_full)

print(f"State after H⊗H → R_ZZ(γ=π/3):")
print(f"{'State':>6} | {'Amplitude':>20} | {'|Amplitude|²':>14} | {'Phase (deg)':>12}")
print("-" * 60)
labels = ['|00⟩','|01⟩','|10⟩','|11⟩']
for lbl, amp in zip(labels, sv.data):
    prob = abs(amp)**2
    phase_deg = np.degrees(np.angle(amp))
    cut = 'CUT' if lbl in ['|01⟩','|10⟩'] else 'same'
    print(f"{lbl:>6} | {amp:>20.4f} | {prob:>14.4f} | {phase_deg:>10.2f}°  ({cut})")

print("\n→ Probabilities are still equal: R_ZZ only changes phases!")
print("→ Cut states (|01⟩,|10⟩) have different phase than non-cut states.")

---
### ✏️ Exercise 4.1 — Verify R_ZZ for all 4 basis states

For each basis state $|00\rangle, |01\rangle, |10\rangle, |11\rangle$:
1. Apply the CNOT–$R_Z(\gamma)$–CNOT circuit and read the statevector
2. Verify that $|01\rangle$ and $|10\rangle$ get phase $+\gamma/2$, while $|00\rangle$ and $|11\rangle$ get phase $-\gamma/2$
3. Why does $R_{ZZ}$ NOT entangle the qubits? (Hint: check if the output is still a product state)

In [ ]:
# YOUR CODE HERE
gamma = np.pi / 3
basis_circuits = {
    '|00⟩': [],           # no initialization = |00⟩
    '|01⟩': ['x1'],      # X on qubit 1
    '|10⟩': ['x0'],      # X on qubit 0
    '|11⟩': ['x0','x1'], # X on both
}

print(f"R_ZZ(γ=π/3) on each basis state:")
print(f"  γ/2 = {np.degrees(gamma/2):.1f}°")
print(f"\n{'Input':>5} | {'Output amplitude':>20} | {'Phase (deg)':>12} | {'Expected':>12}")
print("-" * 60)

for lbl, ops in basis_circuits.items():
    qc = QuantumCircuit(2)
    if 'x0' in ops: qc.x(0)
    if 'x1' in ops: qc.x(1)
    # Apply R_ZZ
    qc.cx(0,1); qc.rz(gamma,1); qc.cx(0,1)
    sv = Statevector(qc)
    # Find non-zero amplitude
    idx = np.argmax(abs(sv.data))
    amp = sv.data[idx]
    phase = np.degrees(np.angle(amp))
    is_cut = lbl in ['|01⟩','|10⟩']
    expected = f"+{np.degrees(gamma/2):.1f}°" if is_cut else f"-{np.degrees(gamma/2):.1f}°"
    print(f"{lbl:>5} | {amp:>20.4f} | {phase:>10.2f}°  | {expected:>12}  {'✓' if abs(phase-float(expected[:-1])) < 0.1 else '✗'}")

print("\n3. R_ZZ does NOT entangle: it's a diagonal matrix.")
print("   Diagonal matrices can't create superposition — they only change phases.")
print("   Entanglement requires off-diagonal terms (like CNOT).")

---
## Part 4: Multiple R_ZZ Gates — Encoding Multiple Edges

In QAOA, each edge of the graph gets its own $R_{ZZ}(\gamma)$ gate.
Multiple $R_{ZZ}$ gates **add their phase contributions**.

In [ ]:
# ── 4.1  Two edges: R_ZZ(01) and R_ZZ(02) on 3 qubits ───────────────────────
# Edges (0,1) and (0,2): phase for |x0x1x2⟩ depends on
#   cut(0,1) = (x0⊕x1) and cut(0,2) = (x0⊕x2)

gamma = np.pi/4
n = 3

qc3 = QuantumCircuit(n)
qc3.h(range(n))       # uniform superposition
# Edge (0,1)
qc3.cx(0,1); qc3.rz(gamma,1); qc3.cx(0,1)
# Edge (0,2)
qc3.cx(0,2); qc3.rz(gamma,2); qc3.cx(0,2)

sv3 = Statevector(qc3)

print(f"3-qubit state after H×3 → R_ZZ(01) → R_ZZ(02), γ=45°:")
print(f"{'Bitstring':>10} | {'Amplitude':>20} | {'P':>6} | {'Phase':>10} | {'cut(01)+cut(02)':>16}")
print("-" * 75)

for i, (bs, amp) in enumerate(zip([format(k,'03b') for k in range(8)], sv3.data)):
    prob = abs(amp)**2
    phase_deg = np.degrees(np.angle(amp))
    cut01 = int(bs[0]) ^ int(bs[1])
    cut02 = int(bs[0]) ^ int(bs[2])
    total_cut = cut01 + cut02
    print(f"{bs:>10} | {amp:>20.4f} | {prob:>6.4f} | {phase_deg:>8.2f}° | {total_cut:>16}")

print("\nObservation: bitstrings with higher total cut value have larger phase.")

In [ ]:
# ── 4.2  Visualize: phase proportional to cut value ───────────────────────────
data = []
for i, (bs, amp) in enumerate(zip([format(k,'03b') for k in range(8)], sv3.data)):
    cut01 = int(bs[0]) ^ int(bs[1])
    cut02 = int(bs[0]) ^ int(bs[2])
    phase = np.degrees(np.angle(amp))
    data.append((bs, cut01+cut02, phase))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

bitstrings  = [d[0] for d in data]
cut_vals    = [d[1] for d in data]
phases      = [d[2] for d in data]
colors      = plt.cm.RdYlGn([(c/2) for c in cut_vals])

ax1.bar(bitstrings, cut_vals, color=colors)
ax1.set_xlabel('Bitstring'); ax1.set_ylabel('Total cut value')
ax1.set_title('Cut values of all 3-qubit bitstrings\n(edges 0-1 and 0-2)')
ax1.tick_params(axis='x', rotation=45)

ax2.bar(bitstrings, phases, color=colors)
ax2.set_xlabel('Bitstring'); ax2.set_ylabel('Phase (degrees)')
ax2.set_title('Phase after R_ZZ(01) + R_ZZ(02)\n(proportional to cut value)')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()
print("Phase is proportional to cut value — this is the 'Judges Scores' step!")

---
## ⭐ Optional: Run R_ZZ on Real IBM Quantum Hardware

The following cells require an IBM Quantum account.
Go to [quantum.ibm.com](https://quantum.ibm.com) to get your free API token.

In [ ]:
# ── Optional: IBM Quantum Setup ───────────────────────────────────────────────
USE_REAL_HARDWARE = False   # Set to True if you have an IBM Quantum account

if USE_REAL_HARDWARE:
    try:
        from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
        # Save your account (only needed once):
        # QiskitRuntimeService.save_account(channel='ibm_quantum', token='YOUR_TOKEN')
        service = QiskitRuntimeService()
        backend = service.least_busy(operational=True, simulator=False, min_num_qubits=2)
        print(f"Using backend: {backend.name}")
    except Exception as e:
        print(f"IBM Quantum not available: {e}")
        USE_REAL_HARDWARE = False
else:
    print("Running on Aer simulator. Set USE_REAL_HARDWARE=True to use IBM Quantum.")

In [ ]:
# ── R_ZZ circuit for hardware comparison ─────────────────────────────────────
# Simple circuit: H⊗H → R_ZZ(π/2) → measure
gamma_hw = np.pi/2

qc_hw = QuantumCircuit(2, 2)
qc_hw.h(0); qc_hw.h(1)
qc_hw.cx(0,1); qc_hw.rz(gamma_hw,1); qc_hw.cx(0,1)
qc_hw.measure([0,1],[0,1])

print("Hardware test circuit:")
print(qc_hw.draw('text'))

# Simulator result
sim_result = simulator.run(transpile(qc_hw, simulator), shots=1000).result()
sim_counts = sim_result.get_counts()
print(f"\nSimulator: {sim_counts}")
print("(Should be ~equal for all 4 outcomes — R_ZZ doesn't change probabilities)")

if USE_REAL_HARDWARE:
    from qiskit_ibm_runtime import SamplerV2 as Sampler
    sampler = Sampler(backend)
    hw_qc = transpile(qc_hw, backend)
    hw_job = sampler.run([hw_qc], shots=1000)
    hw_result = hw_job.result()
    hw_counts = hw_result[0].data.c.get_counts()
    print(f"\nReal hardware: {hw_counts}")
    print("Note: Imperfect due to noise — this is the NISQ reality!")

---
### ✏️ Exercise 4.2 — Build the R_ZZ for a Triangle Graph

Consider a 3-node triangle graph with edges: `(0,1), (1,2), (0,2)`

1. Build a quantum circuit that applies uniform superposition, then $R_{ZZ}(\gamma)$ for **all three edges**
2. For $\gamma = \pi/4$, print the statevector and verify the phase assigned to each bitstring
3. Which bitstrings have the highest cut value for this triangle?
4. Do the phases correctly reflect the cut values?

In [ ]:
# YOUR CODE HERE
gamma = np.pi / 4
triangle_edges = [(0,1), (1,2), (0,2)]

qc_tri = QuantumCircuit(3)
qc_tri.h([0,1,2])  # uniform superposition

# Apply R_ZZ for each edge
for u, v in triangle_edges:
    qc_tri.cx(u, v)
    qc_tri.rz(gamma, v)
    qc_tri.cx(u, v)

sv_tri = Statevector(qc_tri)

def cut_val_triangle(bs, edges):
    return sum(1 for u,v in edges if bs[u] != bs[v])

print(f"Triangle graph (edges {triangle_edges}) — γ=π/4=45°")
print(f"\n{'Bitstring':>10} | {'Phase':>10} | {'Cut value':>10} | {'Expected phase':>15}")
print("-" * 55)

for bs, amp in zip([format(k,'03b') for k in range(8)], sv_tri.data):
    phase = np.degrees(np.angle(amp))
    cv    = cut_val_triangle(bs, triangle_edges)
    # Expected phase depends on parity of each edge
    expected_phase = sum(
        (np.degrees(gamma/2) if bs[u]!=bs[v] else -np.degrees(gamma/2))
        for u,v in triangle_edges
    )
    print(f"{bs:>10} | {phase:>8.2f}° | {cv:>10} | {expected_phase:>13.2f}°")

print("\nMax cut for triangle = 2 (can't cut all 3 edges — odd cycle!)")
print("Bitstrings with cut=2: 001, 010, 100, 110, 101, 011")

---
## ✅ Lab 4 Summary

| Gate | Matrix | Effect |
|------|--------|--------|
| CNOT | $\begin{bmatrix}1&0&0&0\\0&1&0&0\\0&0&0&1\\0&0&1&0\end{bmatrix}$ | Flips target when control=1; creates entanglement |
| $Z\otimes Z$ | $\text{diag}(+1,-1,-1,+1)$ | Signs: same group=+1, diff group=-1 |
| $R_{ZZ}(\gamma)$ | $\text{diag}(e^{-i\gamma/2},e^{+i\gamma/2},e^{+i\gamma/2},e^{-i\gamma/2})$ | Phase depends on whether edge is cut |
| Decomposition | CNOT → $R_Z(\gamma)$ → CNOT | Hardware-friendly circuit |

**Key insight:** Each graph edge $(i,j)$ maps to one $R_{ZZ}(\gamma)$ gate. Multiple gates **accumulate** phases proportionally to the total cut value. This is QAOA Step 2 (Judges' Scores) generalized to all edges.

## 🔭 Preview of Lab 5
Next: **Encoding Max-Cut: The Full Cost Hamiltonian** — we'll build the complete cost operator $U_C(\gamma)$ for all 6 edges of the 5-node graph and verify it assigns correct phases.

---
*QOS Lab 4 | Prof. Chansu Yu | Cleveland State University*